In [ ]:
# =============================================================================
# 0. 安裝必要套件
# =============================================================================
!pip install yfinance pandas numpy google-generativeai plotly gradio requests \
    langchain langchain-google-genai langgraph chromadb beautifulsoup4 \
    langchain-community langchain-core duckduckgo-search
!pip install -U ddgs

import pandas as pd
import yfinance as yf
import numpy as np
import requests
import time
import json
import random
from datetime import datetime, timedelta
from typing import Annotated, List, Dict, Any, Optional, TypedDict, Literal
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import gradio as gr
import google.generativeai as genai
from google.colab import userdata
from functools import wraps

# LangChain & LangGraph Imports
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import HumanMessage, SystemMessage, RemoveMessage, BaseMessage
from langchain_core.tools import tool
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langgraph.graph import StateGraph, END, START
from langgraph.prebuilt import ToolNode
from langchain_community.tools import DuckDuckGoSearchRun # 替代 Google News Scraper 以確保穩定性

  Using cached ddgs-9.10.0-py3-none-any.whl.metadata (12 kB)
  Using cached fake_useragent-2.2.0-py3-none-any.whl.metadata (17 kB)
  Using cached socksio-1.0.0-py3-none-any.whl.metadata (6.1 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.3/40.3 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.7/161.7 kB 6.2 MB/s eta 0:00:00


/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


In [ ]:
# =============================================================================
# 1. 配置與資料庫
# =============================================================================

# Gemini API 設定
try:
    GEMINI_API_KEY = userdata.get('GEMINI_API_KEY_1')
    genai.configure(api_key=GEMINI_API_KEY)
    print("✅ API Key 已設定")
except Exception as e:
    print(f"❌ API 配置失敗: {e}")
    # 為了讓 LangChain 也能用，設定環境變數
    import os
    os.environ["GOOGLE_API_KEY"] = GEMINI_API_KEY if 'GEMINI_API_KEY' in locals() else ""

# K線型態資料庫
K_LINE_PATTERNS_DATABASE = {
    "單日線形K棒": [
        {
            "名稱": "一字線",
            "定義": "開盤價、收盤價、最高價、最低價在同價位上。",
            "圖形": "─",
            "態勢": "停或跌停",
            "可能策略": "股價極強勢或極弱勢，需注意位階。"
        },
        {
            "名稱": "T字線",
            "定義": "開盤價、收盤價、最高價在相同價位上。",
            "圖形": "T",
            "態勢": "支撐力道強勁",
            "可能策略": "1. 低位階：底部支撐訊號。\n2. 初期上漲：支撐訊號。\n3. 注意急漲是否為拉高出貨訊號。"
        },
        {
            "名稱": "墓碑線",
            "定義": "開盤價、收盤價、最低價在相同價位上。",
            "圖形": "┴",
            "態勢": "賣壓力道強勁",
            "可能策略": "1. 高位階：頭部壓力訊號。\n2. 初期下跌：壓力訊號。"
        },
        {
            "名稱": "吊人線",
            "定義": "實體線短而下影線長。",
            "圖形": "T (類似)",
            "態勢": "支撐力道強勁",
            "可能策略": "下降末端出現為買進訊號。"
        },
        {
            "名稱": "十字線",
            "定義": "收盤價等於開盤價，上下影線相當。",
            "圖形": "+",
            "態勢": "多空勢均力敵",
            "可能策略": "行情膠著，多空即將表態。"
        },
        {
            "名稱": "紡錘線",
            "定義": "實體線短而留上下影線。",
            "圖形": "┿",
            "態勢": "多空勢均力敵",
            "可能策略": "1. 急漲後漲勢無力。\n2. 急跌後賣壓趨緩。"
        },
        {
            "名稱": "高浪線",
            "定義": "實體線短且上下影線長。",
            "圖形": "┿ (影線更長)",
            "態勢": "多空交戰激烈",
            "可能策略": "適合當沖。"
        }
    ],
    "雙日線形K棒": [
        {
            "名稱": "烏雲罩頂",
            "定義": "當日黑K吃掉昨日紅K一半或以上。",
            "圖形": "📈📉 (黑K覆蓋紅K)",
            "態勢": "氣勢弱 (高位階)",
            "可能策略": "1. 上升趨勢末端出現，當天收盤前賣出。\n2. 整理或反轉型態。"
        },
        {
            "名稱": "貫穿型態",
            "定義": "當日紅K吃掉昨日黑K一半或以上。",
            "圖形": "📉📈 (紅K覆蓋黑K)",
            "態勢": "氣勢強 (低位階)",
            "可能策略": "1. 下降趨勢末端出現，當天收盤前買進。\n2. 整理或反轉型態。"
        },
        {
            "名稱": "陰包陽",
            "定義": "當日黑K全吃昨日紅K。",
            "圖形": "📈📉 (黑K完全覆蓋紅K)",
            "態勢": "氣勢弱 (高位階)",
            "可能策略": "1. 上升趨勢末端。\n2. 暴量長黑結束多頭。\n當天收盤前賣出。"
        },
        {
            "名稱": "陽包陰",
            "定義": "當日紅K全吃昨天黑K。",
            "圖形": "📉📈 (紅K完全覆蓋黑K)",
            "態勢": "氣勢強 (低位階)",
            "可能策略": "1. 下降趨勢末端。\n2. 窒息量長紅結束空頭，為最佳買點。"
        },
        {
            "名稱": "母子環抱",
            "定義": "當日K線長度不及昨日紅線長度。",
            "圖形": "📈(+/-)",
            "態勢": "行情整理",
            "可能策略": "1. 上升漲勢告一段落。\n2. 下降跌勢告一段落。\n近期找買點或賣點。"
        },
        {
            "名稱": "跳空缺口",
            "定義": "當日K線長度不及昨日紅線長度。",
            "圖形": "📈(+/-)",
            "態勢": "行情整理",
            "可能策略": "1. 上升漲勢告一段落。\n2. 下降跌勢告一段落。\n近期找買點或賣點。"
        }
    ],
    "三日線形K棒": [
        {
            "名稱": "三兵 (三鴉)",
            "定義": "連續三個交易日收盤價低於開盤價。",
            "圖形": "📉📉📉",
            "態勢": "1. 上漲型態利多鈍化。\n2. 短線下降趨勢確立。",
            "可能策略": "弱勢股反彈減碼。"
        },
        {
            "名稱": "三商 (紅三兵)",
            "定義": "連續三個交易日收盤價高於開盤價。",
            "圖形": "📈📈📈",
            "態勢": "1. 上漲無套牢賣壓。\n2. 短線上升趨勢確立。",
            "可能策略": "強勢股拉回加碼。"
        }
    ],
    "價量關係": [
        {
            "名稱": "價漲量增",
            "定義": "股價上漲，成交量增加。",
            "可能意義": "在相對低基期整理一段時間後出現，可能是股價起漲的表態。"
        },
        {
            "名稱": "價跌量增",
            "定義": "股價下跌，成交量增加。",
            "可能意義": "在相對高基期、股價漲不上去時出現，可能是股價起跌的表態。"
        }
    ]
}

class StockDatabase:
    """管理台灣股票清單與收藏功能"""
    def __init__(self):
        self.all_stocks = []
        self.favorites = set()
        self.load_stock_list()

    def load_stock_list(self):
        try:
            print("🔄 正在從證交所獲取股票列表...")
            url = 'https://www.twse.com.tw/exchangeReport/STOCK_DAY_ALL?response=json'
            response = requests.get(url, timeout=10)
            if response.status_code == 200:
                data = response.json()
                self.all_stocks = [
                    {"code": item[0], "name": item[1], "display": f"{item[0]} - {item[1]}"}
                    for item in data.get('data', []) if item[0].isdigit()
                ]
                print(f"✅ 成功載入 {len(self.all_stocks)} 支股票")
            else:
                raise Exception("API請求失敗")
        except Exception as e:
            print(f"❌ 無法獲取股票列表: {e}，使用備用清單")
            self.all_stocks = [
                {"code": "2330", "name": "台積電", "display": "2330 - 台積電"},
                {"code": "2317", "name": "鴻海", "display": "2317 - 鴻海"},
                {"code": "2454", "name": "聯發科", "display": "2454 - 聯發科"},
            ]

    def search_stocks(self, query):
        if not query or len(query.strip()) == 0: return []
        query = query.strip().upper()
        results = [s["display"] for s in self.all_stocks if query in s["code"] or query in s["name"].upper()]
        return results[:20]

    def add_favorite(self, stock_display):
        if stock_display and " - " in stock_display:
            code = stock_display.split(" - ")[0]
            self.favorites.add(code)

    def remove_favorite(self, code):
        self.favorites.discard(code)

    def get_favorites_list(self):
        fav_list = []
        for stock in self.all_stocks:
            if stock["code"] in self.favorites:
                fav_list.append(stock["display"])
        return fav_list

✅ API Key 已設定


In [ ]:
# =============================================================================
# 2. 原始核心功能函式，繪圖與資料獲取
# =============================================================================

def get_stock_data(ticker):
    """
    使用明確的日期範圍獲取數據
    """
    try:
        end_date = datetime.now()
        start_date = end_date - timedelta(days=400) # 抓取過去一年的資料
        print(f"📥 下載 {ticker} 資料中...")
        df = yf.download(ticker, start=start_date.strftime('%Y-%m-%d'), end=end_date.strftime('%Y-%m-%d'), auto_adjust=True, progress=False)

        if df.empty:
            return pd.DataFrame()

        if isinstance(df.columns, pd.MultiIndex):
            df.columns = [col[0].lower() for col in df.columns]
        else:
            df.columns = [col.lower() for col in df.columns]

        # 確保索引是 Datetime
        df.index = pd.to_datetime(df.index)
        return df
    except Exception as e:
        print(f"❌ 下載資料錯誤: {e}")
        return pd.DataFrame()

def plot_interactive_chart(df, ticker):
    """
    回傳 Plotly Figure 物件
    """
    # 0. 資料前處理
    df = df.dropna(subset=['close'])
    df = df.sort_index()

    # 1. 計算指標
    df['ma5'] = df['close'].rolling(window=5).mean()
    df['ma20'] = df['close'].rolling(window=20).mean()

    # 2. 資料切片，取最近一年
    plot_df = df.tail(242).copy()
    plot_df.index = plot_df.index.strftime('%Y-%m-%d')

    # 3. 建立圖表框架
    fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                        vertical_spacing=0.03,
                        row_heights=[0.7, 0.3])

    # 4. 準備顏色
    colors = ['#d62728' if row['close'] - row['open'] >= 0 else '#2ca02c' for index, row in plot_df.iterrows()]
    line_inc = '#d62728'
    line_dec = '#2ca02c'

    # 5. 加入 K線圖
    fig.add_trace(go.Candlestick(x=plot_df.index,
                                 open=plot_df['open'], high=plot_df['high'], low=plot_df['low'], close=plot_df['close'],
                                 name='K線', increasing_line_color=line_inc, decreasing_line_color=line_dec,
                                 hovertext=[f"日期: {i}<br>開盤: {row['open']:.2f}<br>最高: {row['high']:.2f}<br>最低: {row['low']:.2f}<br>收盤: {row['close']:.2f}" for i, row in plot_df.iterrows()]),
                  row=1, col=1)

    fig.add_trace(go.Scatter(x=plot_df.index, y=plot_df['ma5'], mode='lines', name='5日均線', line=dict(color='orange', width=1.2)), row=1, col=1)
    fig.add_trace(go.Scatter(x=plot_df.index, y=plot_df['ma20'], mode='lines', name='20日均線', line=dict(color='blue', width=1.2)), row=1, col=1)

    # 6. 加入 成交量圖
    fig.add_trace(go.Bar(x=plot_df.index, y=plot_df['volume'], name='成交量', marker_color=colors,
                         hovertext=[f"日期: {i}<br>成交量: {row['volume']}" for i, row in plot_df.iterrows()]),
                  row=2, col=1)

    # 7. 更新 Layout
    fig.update_layout(
        title=dict(text=f'<b>{ticker} 技術分析圖表</b>', x=0.05),
        xaxis_rangeslider_visible=False,
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
        height=650,
        margin=dict(l=50, r=20, t=80, b=20),
        # 【關鍵】設定 unified 模式，讓資訊框合併顯示
        hovermode='x unified',
        paper_bgcolor='white',
        plot_bgcolor='#f8f9fa'
    )

    # 設定 X 軸與 Y 軸的準星線 (Spikes)
    fig.update_xaxes(
        type='category',
        tickmode='auto',
        nticks=15,
        showspikes=True, spikemode='across', spikesnap='cursor', showline=True, spikedash='dash', spikecolor="grey"
    )
    fig.update_yaxes(
        showspikes=True, spikemode='across', spikesnap='cursor', showline=True, spikedash='dash', spikecolor="grey"
    )

    # 讓成交量圖也能顯示準星
    fig.update_xaxes(showspikes=True, spikemode='across', row=2, col=1)

    return fig

In [ ]:
# =============================================================================
# 3. 新增多智能體架構 (Agent Architecture)
# =============================================================================

# 狀態定義
class InvestDebateState(TypedDict):
    bull_history: Annotated[str, "Bullish Conversation history"]
    bear_history: Annotated[str, "Bearish Conversation history"]
    history: Annotated[str, "Conversation history"]
    current_response: Annotated[str, "Latest response"]
    judge_decision: Annotated[str, "Final judge decision"]
    count: Annotated[int, "Length of the current conversation"]
    bull_count: Annotated[int, "Bull round count"]
    bear_count: Annotated[int, "Bear round count"]

class AgentState(TypedDict):
    messages: Annotated[List[BaseMessage], "Conversation messages"]
    company_of_interest: Annotated[str, "Company that we are interested in trading"]
    trade_date: Annotated[str, "What date we are trading at"]
    sender: Annotated[str, "Agent that sent this message"]

    # research step
    market_report: Annotated[str, "Report from the Market Analyst"]
    news_report: Annotated[str, "Report from the News Researcher of current world affairs"]

    # researcher team discussion step
    investment_debate_state: Annotated[InvestDebateState, "Current state of the debate on if to invest or not"]
    investment_plan: Annotated[str, "Plan generated by the Analyst"]

    trader_investment_plan: Annotated[str, "Plan generated by the Trader"]
    final_trade_decision: Annotated[str, "Final decision made by the Trader"]
    _analysis_complete_announced: bool
    _phase: str

# 工具定義

# DuckDuckGo Search 作為 Google News 的替代方案 ，無需額外複雜爬蟲設定
search_tool = DuckDuckGoSearchRun()

class Toolkit:
    @staticmethod
    @tool
    def get_YFin_data(
        symbol: Annotated[str, "ticker symbol of the company"],
        start_date: Annotated[str, "Start date in yyyy-mm-dd format"],
        end_date: Annotated[str, "End date in yyyy-mm-dd format"],
    ) -> str:
        """
        Retrieve the stock price data for a given ticker symbol from Yahoo Finance.
        Args:
            symbol (str): Ticker symbol of the company, e.g. AAPL, 2330.TW
            start_date (str): Start date in yyyy-mm-dd format
            end_date (str): End date in yyyy-mm-dd format
        Returns:
            str: A formatted string containing the stock price data.
        """
        try:
            df = yf.download(symbol, start=start_date, end=end_date, progress=False)
            if df.empty:
                return f"No data found for {symbol}."
            return df.tail(10).to_string() # 返回最近10筆以節省 Token
        except Exception as e:
            return f"Error getting stock data: {e}"

    @staticmethod
    @tool
    def get_stockstats_indicators_report(
        symbol: Annotated[str, "ticker symbol of the company"],
        indicator: Annotated[str, "technical indicator to get the analysis and report of"],
        curr_date: Annotated[str, "The current trading date you are trading on, YYYY-mm-dd"],
        look_back_days: Annotated[int, "how many days to look back"] = 30,
    ) -> str:
        """
        Retrieve stock stats indicators for a given ticker symbol and indicator.
        IMPORTANT: This function accepts only ONE indicator per call. Call this function only once
        per analysis with your most important selected indicator.
        """
        try:
            # 這裡簡單模擬指標計算，實際應用可引入 stockstats 庫
            # 為了單檔執行，我們使用 yfinance 內建計算簡單指標
            end_date_obj = datetime.strptime(curr_date, "%Y-%m-%d")
            start_date_obj = end_date_obj - timedelta(days=look_back_days + 60)
            df = yf.download(symbol, start=start_date_obj.strftime("%Y-%m-%d"), end=curr_date, progress=False)

            if df.empty: return "No data."

            result = ""
            if "sma" in indicator.lower() or "ma" in indicator.lower():
                df['MA'] = df['Close'].rolling(window=20).mean()
                last_ma = df['MA'].iloc[-1]
                result = f"20-day MA for {symbol} is {last_ma:.2f}"
            elif "rsi" in indicator.lower():
                # 簡單 RSI 實作
                delta = df['Close'].diff()
                gain = (delta.where(delta > 0, 0)).rolling(window=14).mean()
                loss = (-delta.where(delta < 0, 0)).rolling(window=14).mean()
                rs = gain / loss
                rsi = 100 - (100 / (1 + rs))
                last_rsi = rsi.iloc[-1]
                result = f"14-day RSI for {symbol} is {last_rsi:.2f}"
            else:
                result = f"Generic price data: Close={df['Close'].iloc[-1]:.2f}"

            return f"## {indicator} Report for {symbol}:\n{result}"
        except Exception as e:
            return f"Error calculating indicator: {e}"

    @staticmethod
    @tool
    def get_google_news(
        query: Annotated[str, "Query to search with"],
        curr_date: Annotated[str, "Curr date in yyyy-mm-dd format"],
    ):
        """
        Retrieve the latest news using DuckDuckGo (simulating Google News).
        """
        try:
            results = search_tool.invoke(f"{query} news {curr_date}")
            return results
        except Exception as e:
            return f"Error searching news: {e}"

    @staticmethod
    @tool
    def get_company_info(
        symbol: Annotated[str, "ticker symbol of the company"],
    ) -> str:
        """
        Retrieve basic company information for a given ticker symbol.
        """
        try:
            t = yf.Ticker(symbol)
            info = t.info
            return f"Name: {info.get('longName')}, Sector: {info.get('sector')}, Summary: {info.get('longBusinessSummary')[:200]}..."
        except Exception as e:
            return f"Error getting company info: {e}"

def create_msg_delete():
    def delete_messages(state):
        # 不使用 RemoveMessage。
        # 直接回傳一個包含新 HumanMessage 的列表。
        # 在沒有設定 reducer 的情況下，這通常會直接覆蓋舊的 messages 列表，達到清除歷史的效果。
        placeholder = HumanMessage(content="分析階段切換：請根據新的上下文繼續。")
        return {"messages": [placeholder]}
    return delete_messages

# 記憶體定義
# 由於 Colab 環境限制，我們使用簡單的 In-Memory 向量檢索或直接 Mock
# 為了完整性，這裡使用一個簡單的 List 儲存並透過規則匹配，若要用 ChromaDB 需額外設定持久化路徑
class FinancialSituationMemory:
    def __init__(self, name, config=None):
        self.memories = []

    def get_memories(self, current_situation, n_matches=1):
        # 簡單回傳預設建議，模擬記憶提取
        return [{
            "recommendation": "根據過去經驗，在高波動市場中應減少部位，觀察 20MA 支撐。",
            "similarity_score": 0.9
        }]

    def add_situations(self, situations_and_advice):
        self.memories.extend(situations_and_advice)

# 智能體節點定義

def create_market_analyst(llm, toolkit):
    tools = [toolkit.get_YFin_data, toolkit.get_stockstats_indicators_report]

    def market_analyst_node(state):
        ticker = state["company_of_interest"]
        today = state["trade_date"]

        # 將 K線資料庫轉為字串以便 LLM 閱讀
        k_line_context = json.dumps(K_LINE_PATTERNS_DATABASE, ensure_ascii=False, indent=2)

        system_text = f"""你是一位金融市場分析師。
**當前任務：請立即針對「{ticker}」在「{today}」的市場表現進行分析。**

請嚴格執行以下流程：
1. 呼叫 `get_YFin_data` 獲取 `{ticker}` 的股價數據。
2. 呼叫 `get_stockstats_indicators_report` 獲取技術指標（建議選用 MA, RSI, MACD 或 Bollinger Bands）。
3. **關鍵分析**：利用以下【K線型態資料庫】比對獲取到的股價數據（開/高/低/收），判斷出現了何種型態（如一字線、墓碑線、紅三兵等），並解讀其多空意涵。

【K線型態資料庫】：
{k_line_context}

**報告要求：**
- 必須包含「K線型態分析」章節，明確指出符合資料庫中的哪種型態。
- 結合技術指標提供綜合判斷。
- 請用中文撰寫，並附上指標整理表格。
- **不要詢問股票代碼，直接開始執行工具。**"""

        # 使用 HumanMessage 封裝指令，並覆蓋掉舊的通用指令
        # 這裡我們只保留這個強指令，忽略之前的對話歷史以免混淆，或者將其放在最前面
        sys_msg = HumanMessage(content=system_text)

        # 為了確保 LLM 知道這是最新指令，我們將其放在 messages 列表的最後面，或者作為唯一的訊息
        # 考慮到 LangGraph 的 message reducer 機制，這裡我們構造一個包含上下文的請求
        # 策略：將 System 指令與 State 訊息合併，確保 System 指令在最後強調執行
        all_messages = [sys_msg] + state["messages"]

        llm_with_tools = llm.bind_tools(tools)
        result = llm_with_tools.invoke(all_messages)

        if result.tool_calls:
            return {"messages": [result], "sender": "Market Analyst"}
        else:
            return {"messages": [result], "market_report": result.content, "sender": "Market Analyst"}
    return market_analyst_node

def create_news_analyst(llm, toolkit):
    tools = [toolkit.get_company_info, toolkit.get_google_news]

    def news_analyst_node(state):
        ticker = state["company_of_interest"]
        today = state["trade_date"]

        system_text = f"""你是一位專業金融新聞研究員。
**當前任務：請立即針對「{ticker}」進行新聞面分析，日期基準為「{today}」。**

請嚴格遵循以下流程：
1. **必須**先呼叫 `get_company_info` 獲取 `{ticker}` 的基本資訊。
2. **必須**呼叫 `get_google_news` 搜尋 `{ticker}` 的近期新聞。

【報告撰寫要求】
請以中文撰寫一份結構化的新聞分析報告，包含：
1. 公司要聞摘要 ({ticker})
2. 所屬產業趨勢
3. 總體經濟影響
4. 重點新聞整理表 (Markdown 表格)
5. 新聞分類與理由
6. 綜合交易觀點 (偏多/偏空/趨勢不明)

**不要詢問股票代碼，直接呼叫工具開始工作。**"""

        sys_msg = HumanMessage(content=system_text)
        all_messages = [sys_msg] + state["messages"]

        llm_with_tools = llm.bind_tools(tools)
        result = llm_with_tools.invoke(all_messages)

        if result.tool_calls:
            return {"messages": [result], "sender": "News Analyst"}
        else:
            return {"messages": [result], "news_report": result.content, "sender": "News Analyst"}
    return news_analyst_node

def create_bull_researcher(llm, memory):
    def bull_node(state):
        debate_state = state["investment_debate_state"]
        history = debate_state.get("history", "")
        market_report = state["market_report"]
        news_report = state["news_report"]
        current_response = debate_state.get("current_response", "")

        prompt_text = f"""您是一位看多分析師。您的任務是建立強有力的、基於證據的論證，強調成長潛力。
可用資源：
市場研究報告：{market_report}
新聞：{news_report}
辯論歷史：{history}
最後的論點：{current_response}

格式：提供重點突出、有力的中文回應 (最多300字)。直接且切中要點。"""

        response = llm.invoke(prompt_text)
        argument = f"看多分析師：{response.content}"

        new_state = {
            "history": history + "\n" + argument,
            "bull_history": debate_state.get("bull_history", "") + "\n" + argument,
            "current_response": argument,
            "count": debate_state.get("count", 0) + 1,
            "bull_count": debate_state.get("bull_count", 0) + 1,
            "bear_count": debate_state.get("bear_count", 0),
        }
        return {"investment_debate_state": new_state, "sender": "Bull Researcher"}
    return bull_node

def create_bear_researcher(llm, memory):
    def bear_node(state):
        debate_state = state["investment_debate_state"]
        history = debate_state.get("history", "")
        market_report = state["market_report"]
        news_report = state["news_report"]
        current_response = debate_state.get("current_response", "")

        prompt_text = f"""您是一位看空分析師。您的目標是提出理由充分的論點，強調風險、挑戰和負面指標。
可用資源：
市場研究報告：{market_report}
新聞：{news_report}
辯論歷史：{history}
最後的論點：{current_response}

格式：提供重點突出、有力的中文回應 (最多300字)。直接且切中要點。"""

        response = llm.invoke(prompt_text)
        argument = f"看空分析師：{response.content}"

        new_state = {
            "history": history + "\n" + argument,
            "bear_history": debate_state.get("bear_history", "") + "\n" + argument,
            "current_response": argument,
            "count": debate_state.get("count", 0) + 1,
            "bull_count": debate_state.get("bull_count", 0),
            "bear_count": debate_state.get("bear_count", 0) + 1,
        }
        return {"investment_debate_state": new_state, "sender": "Bear Researcher"}
    return bear_node

def create_trader(llm, memory):
    def trader_node(state):
        company_name = state["company_of_interest"]
        debate_history = state["investment_debate_state"]["history"]

        prompt_text = f"""基於分析師團隊的綜合分析與多空辯論，這是為{company_name}量身定制的投資計劃。
多空辯論歷史：
{debate_history}

請提供具體的買入、賣出或持有建議。以堅定的決策結束，並始終以「最終交易建議：**買入/持有/賣出**」結束您的回應。請用中文撰寫。"""

        result = llm.invoke(prompt_text)
        return {
            "trader_investment_plan": result.content,
            "final_trade_decision": result.content,
            "sender": "Trader"
        }
    return trader_node

# 條件邏輯

class ConditionalLogic:
    def __init__(self, max_debate_rounds=1):
        self.max_debate_rounds = max_debate_rounds

    def should_continue_market(self, state):
        last_message = state["messages"][-1]
        if hasattr(last_message, "tool_calls") and last_message.tool_calls:
            return "tools_market"
        return "Msg Clear Market"

    def should_continue_news(self, state):
        last_message = state["messages"][-1]
        if hasattr(last_message, "tool_calls") and last_message.tool_calls:
            return "tools_news"
        return "Msg Clear News"

    def should_continue_debate(self, state):
        debate_state = state["investment_debate_state"]
        bull_count = debate_state.get("bull_count", 0)
        bear_count = debate_state.get("bear_count", 0)

        if bull_count >= self.max_debate_rounds and bear_count >= self.max_debate_rounds:
            return "Trader"

        if bull_count <= bear_count:
            return "Bull Researcher"
        else:
            return "Bear Researcher"

# 圖建構

class TradingAgentsGraph:
    def __init__(self):
        # 使用 Gemini 作為 LLM
        self.llm = ChatGoogleGenerativeAI(
            model="gemini-2.0-flash",
            google_api_key=GEMINI_API_KEY,
            temperature=0.5
        )
        self.toolkit = Toolkit()
        self.memory = FinancialSituationMemory("main_memory")
        self.tool_nodes = {
            "market": ToolNode([self.toolkit.get_YFin_data, self.toolkit.get_stockstats_indicators_report]),
            "news": ToolNode([self.toolkit.get_company_info, self.toolkit.get_google_news]),
        }
        self.conditional = ConditionalLogic(max_debate_rounds=1)
        self.graph = self._build_graph()

    def _build_graph(self):
        workflow = StateGraph(AgentState)

        # Nodes
        workflow.add_node("Market Analyst", create_market_analyst(self.llm, self.toolkit))
        workflow.add_node("Msg Clear Market", create_msg_delete())
        workflow.add_node("tools_market", self.tool_nodes["market"])

        workflow.add_node("News Analyst", create_news_analyst(self.llm, self.toolkit))
        workflow.add_node("Msg Clear News", create_msg_delete())
        workflow.add_node("tools_news", self.tool_nodes["news"])

        workflow.add_node("Analysis Phase Checker", lambda state: {"_phase": "debate"})

        workflow.add_node("Bull Researcher", create_bull_researcher(self.llm, self.memory))
        workflow.add_node("Bear Researcher", create_bear_researcher(self.llm, self.memory))
        workflow.add_node("Trader", create_trader(self.llm, self.memory))

        # Edges
        workflow.add_edge(START, "Market Analyst")

        workflow.add_conditional_edges(
            "Market Analyst",
            self.conditional.should_continue_market,
            ["tools_market", "Msg Clear Market"]
        )
        workflow.add_edge("tools_market", "Market Analyst")

        workflow.add_edge("Msg Clear Market", "News Analyst")

        workflow.add_conditional_edges(
            "News Analyst",
            self.conditional.should_continue_news,
            ["tools_news", "Msg Clear News"]
        )
        workflow.add_edge("tools_news", "News Analyst")

        workflow.add_edge("Msg Clear News", "Analysis Phase Checker")
        workflow.add_edge("Analysis Phase Checker", "Bull Researcher")

        workflow.add_conditional_edges(
            "Bull Researcher",
            self.conditional.should_continue_debate,
            {"Bear Researcher": "Bear Researcher", "Trader": "Trader"}
        )
        workflow.add_conditional_edges(
            "Bear Researcher",
            self.conditional.should_continue_debate,
            {"Bull Researcher": "Bull Researcher", "Trader": "Trader"}
        )

        workflow.add_edge("Trader", END)

        return workflow.compile()

    def run(self, company_name, trade_date):
        initial_state = {
            "messages": [HumanMessage(content=f"分析 {company_name}")],
            "company_of_interest": company_name,
            "trade_date": trade_date,
            "investment_debate_state": {
                "count": 0, "bull_count": 0, "bear_count": 0,
                "history": "", "bull_history": "", "bear_history": ""
            }
        }
        return self.graph.invoke(initial_state)

# 初始化 Graph
trading_graph = TradingAgentsGraph()

In [ ]:
# =============================================================================
# 4. 介面邏輯，Gradio 整合
# =============================================================================

stock_db = StockDatabase()

def search_and_add(query):
    """即時搜尋"""
    candidates = stock_db.search_stocks(query)
    return gr.update(choices=candidates, visible=len(candidates) > 0)

def add_to_favorites(candidate_choice):
    """加入收藏"""
    if candidate_choice:
        stock_db.add_favorite(candidate_choice)
        return "", gr.update(choices=[], visible=False), gr.update(choices=stock_db.get_favorites_list())
    return None, None, None

def remove_from_favorites_by_selection(favorites_choice):
    """移除收藏"""
    if favorites_choice and " - " in favorites_choice:
        code = favorites_choice.split(" - ")[0]
        stock_db.remove_favorite(code)
        return gr.update(choices=stock_db.get_favorites_list(), value=None)
    return gr.update(choices=stock_db.get_favorites_list())

def analyze_stock(favorites_choice):
    """
    接收 UI 選擇 -> 呼叫核心函式 -> 回傳給 UI
    """
    if not favorites_choice:
        yield None, "⚠️ 請先從「我的收藏」中選擇一支股票"
        return

    try:
        # 1. 解析代號
        code = favorites_choice.split(" - ")[0]
        ticker = f"{code}.TW"

        # UI 回饋：載入中
        yield None, f"📄 正在載入 {ticker} 資料並啟動多智能體分析..."

        # 2. 獲取資料並繪圖 (保留原本視覺化)
        df = get_stock_data(ticker)
        if df.empty:
            yield None, f"❌ 無法獲取 {ticker} 的資料。"
            return

        fig = plot_interactive_chart(df, ticker)
        yield fig, "🤖 AI 團隊正在工作中：市場分析 -> 新聞分析 -> 多空辯論 -> 交易決策..."

        # 3. 執行 LangGraph 多智能體流程
        today = datetime.now().strftime("%Y-%m-%d")
        final_state = trading_graph.run(ticker, today)

        # 4. 格式化輸出報告
        market_report = final_state.get("market_report", "無市場報告")
        news_report = final_state.get("news_report", "無新聞報告")
        debate_history = final_state["investment_debate_state"].get("history", "無辯論紀錄")
        final_decision = final_state.get("trader_investment_plan", "無最終決策")

        full_report = f"""
# 🏁 {ticker} 多智能體深度分析報告

## 📊 市場分析師報告
{market_report}

---

## 📰 新聞研究員報告
{news_report}

---

## 🗣️ 多空團隊辯論摘要
{debate_history}

---

## ⚖️ 首席交易員最終決策
{final_decision}
        """

        yield fig, full_report

    except Exception as e:
        import traceback
        error_msg = f"❌ 發生錯誤: {str(e)}\n{traceback.format_exc()}"
        yield None, error_msg

🔄 正在從證交所獲取股票列表...
✅ 成功載入 1223 支股票


In [ ]:
# =============================================================================
# 5. 建立 Gradio 介面
# =============================================================================

with gr.Blocks(
    title="🇹🇼 台灣股票技術分析系統 V2 (AI Agents)",
    theme=gr.themes.Soft(primary_hue="blue", secondary_hue="cyan"),
    css="""
    .gradio-container { max-width: 1400px !important; }
    .main-title { text-align: center; color: #1e3a8a; font-size: 2.5em; font-weight: bold; margin-bottom: 10px; }
    .subtitle { text-align: center; color: #64748b; font-size: 1.1em; margin-bottom: 30px; }
    .section-header { background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); color: white; padding: 15px; border-radius: 10px; margin: 20px 0 10px 0; font-size: 1.3em; font-weight: bold; }
    .info-box { background-color: #f0f9ff; border-left: 4px solid #3b82f6; padding: 15px; margin: 10px 0; border-radius: 5px; }
    """
) as demo:

    # 標題區
    gr.Markdown("""
        <div class="main-title">🇹🇼 台灣股票技術分析系統 V2</div>
        <div class="subtitle">結合 K線技術分析 × LangGraph 多智能體協作 (Market/News/Debate/Trader)</div>
    """)

    with gr.Row():
        # 左側：搜尋與收藏
        with gr.Column(scale=1):
            gr.Markdown('<div class="section-header">🔍 股票搜尋</div>')

            search_input = gr.Textbox(
                label="輸入股票代碼或名稱",
                placeholder="例如: 2330 或 台積電",
                lines=1
            )

            candidate_radio = gr.Radio(
                label="候選股票（點選後按「加入收藏」）",
                choices=[],
                visible=False
            )

            add_btn = gr.Button("⭐ 加入收藏", variant="primary", size="lg")

            gr.Markdown('<div class="section-header">⭐ 我的收藏</div>')

            favorites_radio = gr.Radio(
                label="收藏清單（選擇後按「開始分析」）",
                choices=stock_db.get_favorites_list()
            )

            with gr.Row():
                analyze_btn = gr.Button("🚀 啟動 AI 團隊分析", variant="primary", size="lg")
                remove_btn = gr.Button("🗑️ 移除收藏", variant="secondary")

            gr.Markdown("""
                <div class="info-box">
                <strong>💡 使用提示：</strong><br>
                1️⃣ 輸入股票代碼或名稱<br>
                2️⃣ 從候選清單選擇並加入收藏<br>
                3️⃣ 在收藏清單選擇股票<br>
                4️⃣ 點擊「啟動 AI 團隊分析」<br>
                5️⃣ 系統將自動執行：技術面分析 -> 新聞面搜尋 -> 多空對話 -> 最終決策
                </div>
            """)

        # 右側：分析結果
        with gr.Column(scale=2):
            gr.Markdown('<div class="section-header">📊 技術分析與 AI 報告</div>')

            chart_output = gr.Plot(label="K線圖表（最近一年）")

            # Markdown 以呈現 AI 的格式化輸出
            analysis_output = gr.Markdown(label="🤖 AI 團隊深度報告", value="請選擇股票並點擊分析...")

    # 底部說明
    gr.Markdown("""
        ---
        <div style="text-align: center; color: #64748b;">
        <p><strong>⚠️ 免責聲明：</strong>本系統提供的分析僅供參考，不構成投資建議。投資有風險，請謹慎決策。</p>
        <p>💻 技術支援：Gemini AI + LangGraph + yfinance + Plotly</p>
        </div>
    """)

    # 事件綁定
    search_input.change(
        fn=search_and_add,
        inputs=[search_input],
        outputs=[candidate_radio]
    )

    add_btn.click(
        fn=add_to_favorites,
        inputs=[candidate_radio],
        outputs=[search_input, candidate_radio, favorites_radio]
    )

    remove_btn.click(
        fn=remove_from_favorites_by_selection,
        inputs=[favorites_radio],
        outputs=[favorites_radio]
    )

    analyze_btn.click(
        fn=analyze_stock,
        inputs=[favorites_radio],
        outputs=[chart_output, analysis_output]
    )

if __name__ == "__main__":
    demo.launch(debug=True)

/tmp/ipython-input-3000176683.py:5: DeprecationWarning:

The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.

/tmp/ipython-input-3000176683.py:5: DeprecationWarning:

The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.



It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://5d99b72e90962c6351.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


📥 下載 2330.TW 資料中...


/tmp/ipython-input-1843272012.py:83: FutureWarning:

YF.download() has changed argument auto_adjust default to True

/tmp/ipython-input-1843272012.py:58: FutureWarning:

YF.download() has changed argument auto_adjust default to True

/tmp/ipython-input-1843272012.py:58: FutureWarning:

YF.download() has changed argument auto_adjust default to True

/tmp/ipython-input-1843272012.py:83: FutureWarning:

YF.download() has changed argument auto_adjust default to True

/tmp/ipython-input-1843272012.py:58: FutureWarning:

YF.download() has changed argument auto_adjust default to True

ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['2330.TW']: YFPricesMissingError('possibly delisted; no price data found  (1d 2026-01-21 -> 2026-01-21)')
/tmp/ipython-input-1843272012.py:58: FutureWarning:

YF.download() has changed argument auto_adjust default to True

/tmp/ipython-input-1843272012.py:83: FutureWarning:

YF.download() has changed argument auto_adjust default to True

/tmp/ipython-input-18432

Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://5d99b72e90962c6351.gradio.live
